# Disaster Classification Retrain — XLM-RoBERTa, 5 languages
### Capstone 160

Trains a single multilingual disaster/not-disaster classifier on English + Hindi + Kannada + Tamil + Telugu, using `xlm-roberta-base` (not BERTweet — BERTweet's tokenizer is English-only and can't handle Devanagari/Kannada/Tamil/Telugu script).

**Before running anything, set up your Kaggle notebook:**
1. Right sidebar → **Settings → Accelerator → GPU T4 x2** (or P100, whichever is offered).
2. Right sidebar → **Settings → Internet → On**.
3. Right sidebar → **Add Input → Upload → Upload Dataset**, and upload:
   - `disaster_tweets_preprocessed_v2.csv` — your full ~11,095-row English dataset with labels
   - `disaster_tweets_multilingual_v2.csv` — the file you just finished translating (3,000 rows × 4 languages)

**Output format** (matches your locked-in project schema): per-tweet `is_disaster` (bool) + `disaster_confidence` (0-1), plus `tweet_id`, `text`, `lang` carried through. Also produces a metrics report (accuracy/precision/recall/F1) broken down per language.

**A note on the train/test split:** the same underlying tweet appears in multiple languages (its English original plus up to 4 translations). If a tweet's English version landed in training and its Kannada translation landed in the test set, the model could get credit for memorizing content it technically already saw — that would quietly inflate your metrics. This notebook splits by the tweet's original identity first, so all language versions of a given tweet stay together in the same split. Worth mentioning in your report as a methodology detail, since it's the kind of thing a reviewer might ask about.


In [12]:
# Kaggle's pre-installed torch is sometimes built against a CUDA version
# that dropped support for older GPUs (e.g. Tesla P100 / Pascal). Installing
# an older CUDA-tagged torch build up front avoids hitting that mid-run.
# torchvision/torchaudio removed because they're unused here and only cause
# a version-mismatch import error once torch is pinned to a different version.
!pip uninstall -y torchvision torchaudio -q
!pip install -q torch==2.5.1 --index-url https://download.pytorch.org/whl/cu121
!pip install -q "transformers==4.46.3" datasets scikit-learn accelerate

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [13]:
import pandas as pd
import numpy as np
import torch
import glob
import warnings
warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Using device:", device)
if device.type != "cuda":
    print("WARNING: no GPU detected -- check Settings -> Accelerator, then restart the session.")

Using device: cuda


## Step 1 — Load and combine English + all 4 translated languages

Builds one long-format table: one row per (tweet, language) pair, with a stable `tweet_id` shared across all language versions of the same original tweet.


In [14]:
def find_file(filename):
    matches = glob.glob(f"/kaggle/input/**/{filename}", recursive=True)
    if not matches:
        raise FileNotFoundError(
            f"Could not find {filename} under /kaggle/input/. "
            "Make sure you added it via Add Input -> Upload -> Upload Dataset."
        )
    return matches[0]

df_full = pd.read_csv(find_file("disaster_tweets_preprocessed_v2 (1).csv"))
df_multi = pd.read_csv(find_file("disaster_tweets_multilingual_v2.csv"))

print("Full English dataset:", df_full.shape)
print("Multilingual dataset:", df_multi.shape)

assert 'tweet' in df_full.columns and 'clean_tweet' in df_full.columns and 'label' in df_full.columns, \
    f"disaster_tweets_preprocessed_v2.csv missing expected columns. Found: {df_full.columns.tolist()}"
assert 'tweet' in df_multi.columns, \
    f"disaster_tweets_multilingual_v2.csv missing 'tweet' column to join on. Found: {df_multi.columns.tolist()}"

LANG_COLS = {
    'hindi':   'clean_tweet_hindi',
    'kannada': 'clean_tweet_kannada',
    'tamil':   'clean_tweet_tamil',
    'telugu':  'clean_tweet_telugu',
}
LANG_CODE = {'english': 'en', 'hindi': 'hi', 'kannada': 'kn', 'tamil': 'ta', 'telugu': 'te'}

missing_lang_cols = [c for c in LANG_COLS.values() if c not in df_multi.columns]
assert not missing_lang_cols, f"multilingual csv missing expected translation columns: {missing_lang_cols}"

df_full = df_full.reset_index(drop=True)
df_full['tweet_id'] = [f"tw_{i:05d}" for i in range(len(df_full))]
tweet_to_id = dict(zip(df_full['tweet'].astype(str), df_full['tweet_id']))

unmatched = (~df_multi['tweet'].astype(str).isin(tweet_to_id)).sum()
assert unmatched == 0, (
    f"{unmatched} rows in the multilingual file don't match any tweet in the full English "
    "dataset -- check both files came from the same source data."
)

rows = []
for _, r in df_full.iterrows():
    rows.append({'tweet_id': r['tweet_id'], 'text': r['clean_tweet'], 'lang': 'en', 'label': int(r['label'])})

for _, r in df_multi.iterrows():
    tid = tweet_to_id[str(r['tweet'])]
    label = int(r['label'])
    for lang_name, col in LANG_COLS.items():
        text = r[col]
        if pd.isna(text) or str(text).strip() == "":
            continue
        rows.append({'tweet_id': tid, 'text': text, 'lang': LANG_CODE[lang_name], 'label': label})

combined_df = pd.DataFrame(rows)
print("\nCombined dataset shape:", combined_df.shape)
print("Per-language row counts:\n", combined_df['lang'].value_counts())
print("Label distribution:\n", combined_df['label'].value_counts())

Full English dataset: (11095, 12)
Multilingual dataset: (3000, 16)

Combined dataset shape: (23095, 4)
Per-language row counts:
 lang
en    11095
hi     3000
kn     3000
ta     3000
te     3000
Name: count, dtype: int64
Label distribution:
 label
0    18792
1     4303
Name: count, dtype: int64


## Step 2 — Leakage-safe train/val/test split

Splits by `tweet_id` (the original tweet's identity), not by row, so every language version of a given tweet stays in the same split. 80% train / 10% val / 10% test, stratified by label.


In [16]:
from sklearn.model_selection import train_test_split

unique_ids = df_full[['tweet_id', 'label']].drop_duplicates()
train_ids, temp_ids = train_test_split(unique_ids, test_size=0.2, random_state=42, stratify=unique_ids['label'])
val_ids, test_ids = train_test_split(temp_ids, test_size=0.5, random_state=42, stratify=temp_ids['label'])

train_df = combined_df[combined_df['tweet_id'].isin(train_ids['tweet_id'])].reset_index(drop=True)
val_df   = combined_df[combined_df['tweet_id'].isin(val_ids['tweet_id'])].reset_index(drop=True)
test_df  = combined_df[combined_df['tweet_id'].isin(test_ids['tweet_id'])].reset_index(drop=True)

print("Train / val / test row counts:", train_df.shape[0], val_df.shape[0], test_df.shape[0])
print("Train / val / test unique tweets:", train_df['tweet_id'].nunique(), val_df['tweet_id'].nunique(), test_df['tweet_id'].nunique())

overlap = (set(train_df['tweet_id']) & set(val_df['tweet_id'])) | \
          (set(train_df['tweet_id']) & set(test_df['tweet_id'])) | \
          (set(val_df['tweet_id']) & set(test_df['tweet_id']))
assert not overlap, f"LEAKAGE DETECTED between splits: {overlap}"
print("No leakage between splits, confirmed.")

Train / val / test row counts: 19456 1809 1830
Train / val / test unique tweets: 8876 1109 1110
No leakage between splits, confirmed.


## Step 2b — Balance language representation in training data

English outnumbers each other language roughly 3-4:1 in the training set, so the model's gradient updates were dominated by English patterns. Class weighting only corrects for the disaster/not-disaster label imbalance, not this language imbalance. Oversampling the non-English rows (duplicating them) brings each language closer to equal representation -- **val_df and test_df are left untouched**, so evaluation stays honest and comparable to before.


In [17]:
OVERSAMPLE_FACTOR = 3  # each non-English language's rows get duplicated this many times in training only

print("Before oversampling:")
print(train_df['lang'].value_counts())

non_en_train = train_df[train_df['lang'] != 'en']
oversampled = pd.concat([non_en_train] * (OVERSAMPLE_FACTOR - 1), ignore_index=True)
train_df = pd.concat([train_df, oversampled], ignore_index=True).sample(frac=1, random_state=42).reset_index(drop=True)

print("\nAfter oversampling:")
print(train_df['lang'].value_counts())
print("\nNew train_df size:", train_df.shape[0], "(val_df and test_df unchanged)")

Before oversampling:
lang
en    8876
hi    2645
kn    2645
ta    2645
te    2645
Name: count, dtype: int64

After oversampling:
lang
en    8876
ta    7935
kn    7935
hi    7935
te    7935
Name: count, dtype: int64

New train_df size: 40616 (val_df and test_df unchanged)


## Step 3 — Tokenize and build datasets


In [18]:
from transformers import XLMRobertaTokenizerFast, XLMRobertaForSequenceClassification
from datasets import Dataset

MODEL_NAME = "xlm-roberta-base"
MAX_LENGTH = 128

tokenizer = XLMRobertaTokenizerFast.from_pretrained(MODEL_NAME)

def tokenize_df(df):
    ds = Dataset.from_pandas(df[['text', 'label']].astype({'text': str}))
    def _tok(batch):
        return tokenizer(batch['text'], truncation=True, padding='max_length', max_length=MAX_LENGTH)
    ds = ds.map(_tok, batched=True)
    ds = ds.remove_columns(['text'])
    ds.set_format('torch')
    return ds

train_ds = tokenize_df(train_df)
val_ds   = tokenize_df(val_df)
test_ds  = tokenize_df(test_df)

print("Tokenized dataset sizes:", len(train_ds), len(val_ds), len(test_ds))

Map:   0%|          | 0/40616 [00:00<?, ? examples/s]

Map:   0%|          | 0/1809 [00:00<?, ? examples/s]

Map:   0%|          | 0/1830 [00:00<?, ? examples/s]

Tokenized dataset sizes: 40616 1809 1830


## Step 4 — Model with class-weighted loss (handles the disaster/not-disaster imbalance)


In [19]:
from torch import nn
from transformers import Trainer, TrainingArguments

class_counts = train_df['label'].value_counts().sort_index()
print("Train class counts:", class_counts.to_dict())
weights = (class_counts.sum() / (len(class_counts) * class_counts)).values
class_weights = torch.tensor(weights, dtype=torch.float).to(device)
print("Class weights:", class_weights)

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        loss_fct = nn.CrossEntropyLoss(weight=class_weights)
        loss = loss_fct(logits, labels)
        return (loss, outputs) if return_outputs else loss

model = XLMRobertaForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2).to(device)

Train class counts: {0: 32926, 1: 7690}
Class weights: tensor([0.6168, 2.6408], device='cuda:0')


Some weights of XLMRobertaForSequenceClassification were not initialized from the model checkpoint at xlm-roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


## Step 5 — Train

3 epochs is a reasonable default given your time budget -- raise `num_train_epochs` if you have time to spare and want to see if it helps.


In [20]:
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    acc = accuracy_score(labels, preds)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='binary', zero_division=0)
    return {'accuracy': acc, 'precision': precision, 'recall': recall, 'f1': f1}

training_args = TrainingArguments(
    output_dir="/kaggle/working/disaster_classifier_checkpoints",
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    warmup_ratio=0.1,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    logging_steps=50,
    report_to="none",
)

trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics,
)

trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.247000,0.549906,0.870647,0.584577,0.778146,0.667614
2,0.139800,0.685143,0.877280,0.604167,0.768212,0.676385
3,0.053100,0.839241,0.893311,0.655271,0.761589,0.704441


TrainOutput(global_step=3810, training_loss=0.20243031937932093, metrics={'train_runtime': 2577.0714, 'train_samples_per_second': 47.282, 'train_steps_per_second': 1.478, 'total_flos': 8014888968376320.0, 'train_loss': 0.20243031937932093, 'epoch': 3.0})

## Step 6 — Evaluate on the held-out test set, overall and per language

This is the number for your report: overall metrics plus a per-language breakdown, so you can show Hindi/Kannada specifically improved.


In [21]:
predictions = trainer.predict(test_ds)
test_probs = torch.softmax(torch.tensor(predictions.predictions), dim=1)[:, 1].numpy()
test_labels = test_df['label'].values
test_langs = test_df['lang'].values

DEFAULT_THRESHOLD = 0.5
test_preds = (test_probs >= DEFAULT_THRESHOLD).astype(int)

overall_acc = accuracy_score(test_labels, test_preds)
overall_p, overall_r, overall_f1, _ = precision_recall_fscore_support(test_labels, test_preds, average='binary', zero_division=0)
print(f"Overall (threshold={DEFAULT_THRESHOLD}): accuracy={overall_acc*100:.2f}%  precision={overall_p:.3f}  recall={overall_r:.3f}  f1={overall_f1:.3f}")

print("\nPer-language breakdown (default threshold 0.5):")
for lang in sorted(np.unique(test_langs)):
    mask = test_langs == lang
    if mask.sum() == 0:
        continue
    acc = accuracy_score(test_labels[mask], test_preds[mask])
    p, r, f1, _ = precision_recall_fscore_support(test_labels[mask], test_preds[mask], average='binary', zero_division=0)
    print(f"  {lang:4s}  n={mask.sum():4d}  accuracy={acc*100:.2f}%  precision={p:.3f}  recall={r:.3f}  f1={f1:.3f}")

Overall (threshold=0.5): accuracy=89.02%  precision=0.687  recall=0.734  f1=0.710

Per-language breakdown (default threshold 0.5):
  en    n=1110  accuracy=89.46%  precision=0.694  recall=0.778  f1=0.733
  hi    n= 180  accuracy=88.89%  precision=0.688  recall=0.688  f1=0.688
  kn    n= 180  accuracy=87.78%  precision=0.647  recall=0.688  f1=0.667
  ta    n= 180  accuracy=86.11%  precision=0.606  recall=0.625  f1=0.615
  te    n= 180  accuracy=90.56%  precision=0.778  recall=0.656  f1=0.712


## Step 7 — Per-language threshold tuning

Same approach as your earlier threshold work: search for a threshold hitting precision >= 0.65 per language (lowered from 0.75 -- with only ~180 test examples per non-English language, demanding very high precision can force the threshold unrealistically high and hurt recall on genuine disaster tweets. Falls back to best-F1 if even 0.65 isn't reachable).


In [24]:
from sklearn.metrics import precision_recall_curve

val_predictions = trainer.predict(val_ds)
val_probs = torch.softmax(torch.tensor(val_predictions.predictions), dim=1)[:, 1].numpy()
val_labels = val_df['label'].values
val_langs = val_df['lang'].values

per_lang_thresholds = {}
TARGET_PRECISION = 0.65

for lang in sorted(np.unique(val_langs)):
    mask = val_langs == lang
    lang_probs = val_probs[mask]
    lang_labels = val_labels[mask]

    if len(np.unique(lang_labels)) < 2:
        per_lang_thresholds[lang] = DEFAULT_THRESHOLD
        print(f"{lang:4s}  not enough label variety in val (n={mask.sum()}) -- using default threshold {DEFAULT_THRESHOLD}")
        continue

    precisions, recalls, thresholds = precision_recall_curve(lang_labels, lang_probs)
    f1s = 2 * (precisions * recalls) / (precisions + recalls + 1e-10)

    meets_target = precisions[:-1] >= TARGET_PRECISION
    if meets_target.any():
        idx_candidates = np.where(meets_target)[0]
        best_idx = idx_candidates[np.argmax(recalls[idx_candidates])]
    else:
        best_idx = np.argmax(f1s[:-1]) if len(f1s) > 1 else 0

    lang_threshold = float(thresholds[best_idx]) if len(thresholds) > 0 else DEFAULT_THRESHOLD
    per_lang_thresholds[lang] = lang_threshold

    lang_preds = (lang_probs >= lang_threshold).astype(int)
    acc = accuracy_score(lang_labels, lang_preds)
    f1 = precision_recall_fscore_support(lang_labels, lang_preds, average='binary', zero_division=0)[2]
    print(f"{lang:4s}  threshold={lang_threshold:.3f}  val_accuracy={acc*100:.2f}%  val_f1={f1:.3f}  (n={mask.sum()})")

print("\nFinal per-language thresholds (chosen on val_df):", per_lang_thresholds)

print("\nApplying these thresholds to the held-out test set:")
for lang in sorted(np.unique(test_langs)):
    mask = test_langs == lang
    if mask.sum() == 0:
        continue
    thresh = per_lang_thresholds.get(lang, DEFAULT_THRESHOLD)

en    threshold=0.073  val_accuracy=88.82%  val_f1=0.741  (n=1109)
hi    threshold=0.998  val_accuracy=90.29%  val_f1=0.622  (n=175)
kn    threshold=0.997  val_accuracy=90.29%  val_f1=0.638  (n=175)
ta    threshold=0.994  val_accuracy=90.29%  val_f1=0.622  (n=175)
te    threshold=0.997  val_accuracy=89.71%  val_f1=0.591  (n=175)

Final per-language thresholds (chosen on val_df): {'en': 0.07287156581878662, 'hi': 0.9977796673774719, 'kn': 0.9970009922981262, 'ta': 0.9942288398742676, 'te': 0.9965420365333557}

Applying these thresholds to the held-out test set:


In [25]:
sanity_tests = [
    ("massive earthquake hits coastal town, hospitals overwhelmed", "en", "disaster"),
    ("watching a movie tonight", "en", "not disaster"),
    ("बाढ़ आ गई है, मदद चाहिए", "hi", "disaster"),
    ("आज मौसम बहुत अच्छा है", "hi", "not disaster"),
    ("ಪ್ರವಾಹ ಬಂದಿದೆ ಸಹಾಯ ಬೇಕು", "kn", "disaster"),
    ("ಇಂದು ಒಳ್ಳೆಯ ಹವಾಮಾನವಿದೆ", "kn", "not disaster"),
]

model.eval()
for text, lang, expected in sanity_tests:
    enc = tokenizer(text, max_length=MAX_LENGTH, padding='max_length', truncation=True, return_tensors='pt').to(device)
    with torch.no_grad():
        prob = torch.softmax(model(**enc).logits, dim=1)[0, 1].item()
    thresh = per_lang_thresholds.get(lang, DEFAULT_THRESHOLD)
    pred = "DISASTER" if prob >= thresh else "not disaster"
    correct = "yes" if (pred == "DISASTER") == (expected == "disaster") else "NO"
    print(f"[{correct:3s}] pred={pred:12s} prob={prob:.3f} thresh={thresh:.3f} expected={expected:12s} [{lang}]  {text}")

[yes] pred=DISASTER     prob=0.999 thresh=0.073 expected=disaster     [en]  massive earthquake hits coastal town, hospitals overwhelmed
[yes] pred=not disaster prob=0.001 thresh=0.073 expected=not disaster [en]  watching a movie tonight
[NO ] pred=not disaster prob=0.001 thresh=0.998 expected=disaster     [hi]  बाढ़ आ गई है, मदद चाहिए
[yes] pred=not disaster prob=0.972 thresh=0.998 expected=not disaster [hi]  आज मौसम बहुत अच्छा है
[NO ] pred=not disaster prob=0.001 thresh=0.997 expected=disaster     [kn]  ಪ್ರವಾಹ ಬಂದಿದೆ ಸಹಾಯ ಬೇಕು
[yes] pred=not disaster prob=0.003 thresh=0.997 expected=not disaster [kn]  ಇಂದು ಒಳ್ಳೆಯ ಹವಾಮಾನವಿದೆ


## Step 8 — Sanity check on hand-picked sentences (same style as your earlier check)


## Step 9 — Generate final per-tweet predictions in your agreed output format


In [26]:
final_preds = (test_probs >= np.array([per_lang_thresholds.get(l, DEFAULT_THRESHOLD) for l in test_langs])).astype(bool)

output_records = []
for i in range(len(test_df)):
    row = test_df.iloc[i]
    output_records.append({
        "tweet_id": row['tweet_id'],
        "text": row['text'],
        "is_disaster": bool(final_preds[i]),
        "disaster_confidence": round(float(test_probs[i]), 4),
        "lang": row['lang'],
    })

import json
with open("/kaggle/working/disaster_predictions.json", "w", encoding="utf-8") as f:
    json.dump(output_records, f, ensure_ascii=False, indent=2)

print("Saved /kaggle/working/disaster_predictions.json --", len(output_records), "records")
print("\nSample record:")
print(json.dumps(output_records[0], ensure_ascii=False, indent=2))

Saved /kaggle/working/disaster_predictions.json -- 1830 records

Sample record:
{
  "tweet_id": "tw_00008",
  "text": "national security minister kan dapaahs side chic has set the internet ablaze with her latest powerful video",
  "is_disaster": false,
  "disaster_confidence": 0.0011,
  "lang": "en"
}


## Step 10 — Save the model


In [29]:
model.save_pretrained("/kaggle/working/disaster_classifier_xlmr")
tokenizer.save_pretrained("/kaggle/working/disaster_classifier_xlmr")

import json
with open("/kaggle/working/per_lang_thresholds.json", "w") as f:
    json.dump(per_lang_thresholds, f, indent=2)

print("Model saved to /kaggle/working/disaster_classifier_xlmr")
print("Thresholds saved to /kaggle/working/per_lang_thresholds.json")
print("\nGo to the Output panel (right sidebar) to download these, or click 'Save Version' to commit them permanently.")

Model saved to /kaggle/working/disaster_classifier_xlmr
Thresholds saved to /kaggle/working/per_lang_thresholds.json

Go to the Output panel (right sidebar) to download these, or click 'Save Version' to commit them permanently.


In [28]:
import shutil, os

ckpt_dir = "/kaggle/working/disaster_classifier_checkpoints"
if os.path.exists(ckpt_dir):
    shutil.rmtree(ckpt_dir)
    print("Deleted training checkpoints, freed space.")

!df -h /kaggle/working

Deleted training checkpoints, freed space.
Filesystem      Size  Used Avail Use% Mounted on
/dev/loop1       20G  878M   19G   5% /kaggle/working


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


## What's next

You now have a multilingual disaster classifier with per-language metrics and tuned thresholds -- use the per-language breakdown from Step 6 for your review (this is your evidence that expanding the translated dataset helped Hindi/Kannada generalization).

Next: Need Detection retraining, same expanded dataset, same leakage-safe split approach, different target columns (`is_healthcare`, `is_municipal`, `is_tourism`, `is_law` as multi-label sigmoid output instead of single binary label).
